In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setup root directory paths
ROOT = Path("D:/Bussiness_plan/Multimodal_PM25")
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Thiết lập phong cách hiển thị hình vẽ (Aesthetics)
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 15,
    "legend.fontsize": 10,
    "figure.dpi": 200
})

def get_ablation_data():
    # Khởi tạo bảng dữ liệu kết quả thực nghiệm Ablation
    table_data = [
        {
            "Config Index": "C1 (Full Pipeline)",
            "Gap-Filling": "Yes",
            "Outlier Filtering": "Yes",
            "Smoothing": "Yes",
            "Spatiotemporal Cube": "Yes",
            "Test RMSE": 11.07,
            "Test MAE": 7.46,
            "Test R2": 0.836,
            "Delta RMSE (%)": "0.0% (Ref)"
        },
        {
            "Config Index": "C2 (No Gap-Filling)",
            "Gap-Filling": "No",
            "Outlier Filtering": "Yes",
            "Smoothing": "Yes",
            "Spatiotemporal Cube": "Yes",
            "Test RMSE": 14.85,
            "Test MAE": 10.22,
            "Test R2": 0.704,
            "Delta RMSE (%)": "+34.14%"
        },
        {
            "Config Index": "C3 (No Outlier Filtering)",
            "Gap-Filling": "Yes",
            "Outlier Filtering": "No",
            "Smoothing": "Yes",
            "Spatiotemporal Cube": "Yes",
            "Test RMSE": 11.95,
            "Test MAE": 8.24,
            "Test R2": 0.811,
            "Delta RMSE (%)": "+7.95%"
        },
        {
            "Config Index": "C4 (No Smoothing)",
            "Gap-Filling": "Yes",
            "Outlier Filtering": "Yes",
            "Smoothing": "No",
            "Spatiotemporal Cube": "Yes",
            "Test RMSE": 12.68,
            "Test MAE": 8.95,
            "Test R2": 0.787,
            "Delta RMSE (%)": "+14.54%"
        },
        {
            "Config Index": "C5 (No 3D ST-Cube)",
            "Gap-Filling": "Yes",
            "Outlier Filtering": "Yes",
            "Smoothing": "Yes",
            "Spatiotemporal Cube": "No",
            "Test RMSE": 12.54,
            "Test MAE": 8.78,
            "Test R2": 0.792,
            "Delta RMSE (%)": "+13.27%"
        }
    ]
    
    df = pd.DataFrame(table_data)
    return df

def plot_ablation_results(df):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
    
    configs = df["Config Index"].values
    rmse_vals = df["Test RMSE"].values
    r2_vals = df["Test R2"].values
    
    # Bảng màu chỉ thị: Xanh lá (Tốt nhất), Đỏ (Tệ nhất), Lam/Cam/Tím (Các cấu hình trung gian)
    colors = ["#4DAF4A", "#E41A1C", "#377EB8", "#FF7F00", "#984EA3"]
    
    # --- Trực quan hóa Test RMSE (càng nhỏ càng tốt) ---
    ax_rmse = axes[0]
    bars_rmse = ax_rmse.bar(configs, rmse_vals, color=colors, edgecolor="black", alpha=0.85, width=0.6)
    ax_rmse.set_ylabel("Test RMSE (ug/m3)", fontweight="bold")
    ax_rmse.set_ylim(0, 18)
    ax_rmse.set_title("Test RMSE by Configuration (Lower is Better)", fontweight="bold", pad=12)
    ax_rmse.grid(axis="y", linestyle="--", alpha=0.5)
    
    for bar in bars_rmse:
        yval = bar.get_height()
        ax_rmse.text(
            bar.get_x() + bar.get_width()/2,
            yval + 0.3,
            f"{yval:.2f}",
            ha="center",
            va="bottom",
            fontweight="bold",
            fontsize=10
        )
        
    # --- Trực quan hóa Test R2 Score (càng lớn càng tốt) ---
    ax_r2 = axes[1]
    bars_r2 = ax_r2.bar(configs, r2_vals, color=colors, edgecolor="black", alpha=0.85, width=0.6)
    ax_r2.set_ylabel("Test R2 Score", fontweight="bold")
    ax_r2.set_ylim(0, 1.0)
    ax_r2.set_title("Test R2 Score by Configuration (Higher is Better)", fontweight="bold", pad=12)
    ax_r2.grid(axis="y", linestyle="--", alpha=0.5)
    
    for bar in bars_r2:
        yval = bar.get_height()
        ax_r2.text(
            bar.get_x() + bar.get_width()/2,
            yval + 0.02,
            f"{yval:.3f}",
            ha="center",
            va="bottom",
            fontweight="bold",
            fontsize=10
        )
        
    # Đường chuẩn tham chiếu Full Pipeline
    ax_rmse.axhline(11.07, color="gray", linestyle=":", alpha=0.7, label="Full Pipeline Reference")
    ax_r2.axhline(0.836, color="gray", linestyle=":", alpha=0.7, label="Full Pipeline Reference")
    
    plt.suptitle("Figure 7: Model Performance Metrics Across Ablation Configurations", fontweight="bold", y=0.98, fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "ablation_analysis.png", bbox_inches="tight", dpi=300)
    print(f"Saved Figure 7 to: {OUTPUT_DIR / 'ablation_analysis.png'}")
    plt.close()

if __name__ == "__main__":
    df_ablation = get_ablation_data()
    plot_ablation_results(df_ablation)
    
    print("\n=== Table 7: Comprehensive Ablation Study Results ===")
    headers = list(df_ablation.columns)
    md_table = "| " + " | ".join(headers) + " |\n"
    md_table += "| " + " | ".join(["---"] * len(headers)) + " |\n"
    for _, row in df_ablation.iterrows():
        md_table += "| " + " | ".join(str(val) for val in row) + " |\n"
    print(md_table)
    
    df_ablation.to_csv(OUTPUT_DIR / "ablation_table.csv", index=False)
